# Chapter 4: ten checks before any chart
The dirty haze export has 63 rows for 56 expected results. The notebook runs the ten checks in order, writes a clean table of current rows and a fault log.

**No code tier.** In a spreadsheet: count rows against samples x tests; sort by sample ID to see duplicates; filter for values that are not plain numbers; filter for blank units; sort by value; compare entered and sampled times; check the ID length; filter blank lot IDs; keep rows where is_current is 1.

Synthetic data from Kestrel Brewing, an invented plant. Copyright 2026 Ankur Napa. All rights reserved. You may use these files for your own learning; do not republish or redistribute them.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = "../data/"

In [ ]:
raw = pd.read_csv(DATA + "ch04_haze_export_raw.csv", keep_default_na=False, dtype=str)
log = []
def note(check, rows, action):
    log.append(dict(check=check, rows=", ".join(rows.row) if len(rows) else "none", action=action))
print("1 row count:", len(raw), "rows for 28 samples x 2 angles =", 28 * 2)
cols = [c for c in raw.columns if c != "row"]
dup = raw[raw.duplicated(subset=cols)]
note("2 duplicate keys", dup, "dropped"); df = raw.drop(dup.index)
print("2 duplicates:", len(dup), "| rows left", len(df))

In [ ]:
num = pd.to_numeric(df.value, errors="coerce")
text = df[num.isna()]; note("3 numbers stored as text", text, "unit stripped")
df["value"] = pd.to_numeric(df.value.str.replace(" EBC", "", regex=False))
blank = df[df.unit == ""]; note("4 one unit per test code", blank, "filled from the test table: EBC"); df.loc[blank.index, "unit"] = "EBC"
spike = df[df.value > 0.5 * 2]; note("5 range against the spec", spike, "sent to the analyst, not fixed")
early = df[df.entered_at < df.sampled_at]; note("6 times in order", early, "sent to the analyst")
bad_id = df[~df.sample_id.str.match(r"^S\d{6}-\d{3}$")]; note("7 ID format", bad_id, "padded")
df["sample_id"] = [s[:8] + s[8:].zfill(3) for s in df.sample_id]
no_key = df[df.lot_id == ""]; note("8 batch key present", no_key, f"{no_key.sample_id.nunique()} samples chased with the shift")
old = df[df.is_current == "0"]; note("9 one current row per sample and test", old, "superseded rows kept in the raw file only")
clean = df[df.is_current == "1"]
odd = clean[(clean.qualifier != "") | (clean.status != "approved")]; note("10 qualifiers kept, approved only", odd, "check still runs")
print(len(clean), "current rows")
pd.DataFrame(log)

In [ ]:
clean.to_csv("ch04_clean.csv", index=False); pd.DataFrame(log).to_csv("ch04_fault_log.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
ax[0].plot(pd.to_numeric(raw.value.str.replace(" EBC", "", regex=False)).clip(upper=0.6), "o", ms=3, color="#2f7f72"); ax[0].set_title("raw export, 63 rows")
ax[1].plot(clean.value.clip(upper=0.6).to_numpy(), "o", ms=3, color="#0b4f45"); ax[1].set_title("current rows, 56 (4.1 held, drawn at 0.6)")
ax[0].set_ylabel("haze (EBC)"); plt.show()

**What to look at.** Compare the fault log with `ch04_planted_faults.csv`. The 4.1 stays in the clean table, held for the analyst: a cleaning script must not guess the decimal. A 0.42 typed for 0.24 passes all ten checks.